# Pipeline Architecture and Error Analysis

This notebook documents the deployed financial risk alert pipeline without retraining any models. It reuses the trained components already defined in `src/`:

- SBERT denoising / headline-body consistency filter
- FinBERT risk classifier
- CRF-based organization extractor
- Template-backed alert generator

The focus here is system behavior: how an article moves from raw news input to a final alert, and how errors at one stage propagate to later stages.

## Full System Architecture

```text
Raw news article (title + body)
        |
        v
Denoising / consistency filter
SBERT cosine(title, body) >= 0.54 ?
        |
        +-- No  -> stop early, article discarded as mismatch / clickbait
        |
        v
Risk classification
FinBERT -> {High Risk, Neutral, Low Risk} + probabilities
        |
        v
Alert decision
Trigger if label == High Risk OR P(High Risk) >= 0.30
        |
        +-- No  -> no alert
        |
        v
Named entity recognition
CRF extracts ORG mentions from title + body
        |
        v
Alert template / generation
LLM generator if token available, otherwise deterministic fallback template
```

### Stage responsibilities

| Stage | Input | Output | Main failure mode | Downstream effect |
|---|---|---|---|---|
| Denoising | Title + body | pass/fail, similarity score | Good article filtered out | Entire article never reaches classifier or alerting |
| Classification | Filtered article | risk label, confidence, class probabilities | False positive / false negative | False alert or missed alert |
| NER | Alerted article text | extracted organizations | Missing or wrong organization spans | Alert has incomplete or misleading targets |
| Alert template | Risk + orgs + snippet | final alert text | Generation or formatting failure | Falls back to deterministic alert text |

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("SENTENCE_TRANSFORMERS_SILENT", "1")

from src.sbert_filter import ClickbaitFilter
from src.finbert_classifier import RiskClassifier
from src.crf_extractor import CRFExtractor
from src.llm_alert_generator import LLMAlertGenerator

SBERT_THRESHOLD = 0.54
HIGH_RISK_ALERT_THRESHOLD = 0.30


In [ ]:
def load_components(use_local=False):
    """Load the trained pipeline components without retraining anything."""
    components = {}
    load_notes = []

    try:
        components["sbert_filter"] = ClickbaitFilter(threshold=SBERT_THRESHOLD)
        load_notes.append("SBERT filter loaded.")
    except Exception as exc:
        components["sbert_filter"] = None
        load_notes.append(f"SBERT unavailable: {exc}")

    try:
        components["risk_classifier"] = RiskClassifier(
            model_path="xuyifei1234/finbert-risk-classifier-v2",
            use_local=use_local,
        )
        load_notes.append("FinBERT risk classifier loaded.")
    except Exception as exc:
        components["risk_classifier"] = None
        load_notes.append(f"FinBERT unavailable: {exc}")

    try:
        components["crf_extractor"] = CRFExtractor(crf_path=str(ROOT / "models" / "crf_org_extractor.pkl"))
        if components["crf_extractor"].crf is None:
            load_notes.append("CRF extractor object created, but no trained CRF weights were found.")
        else:
            load_notes.append("CRF extractor loaded.")
    except Exception as exc:
        components["crf_extractor"] = None
        load_notes.append(f"CRF unavailable: {exc}")

    try:
        components["alert_generator"] = LLMAlertGenerator()
        load_notes.append("LLM alert generator loaded.")
    except Exception as exc:
        components["alert_generator"] = None
        load_notes.append(f"LLM generator unavailable, fallback template will be used: {exc}")

    return components, load_notes


components, load_notes = load_components(use_local=False)
pd.DataFrame({"status": load_notes})

## Article Trace Function

The next cell runs an article through the same staged logic as `src/pipeline.py`, but records intermediate decisions so we can localize failure points instead of only seeing the final alert.

In [ ]:
def fallback_alert_template(title, risk_label, confidence, orgs, high_risk_prob):
    org_str = ", ".join(orgs) if orgs else "No specific organizations identified"
    return (
        f"ALERT: {risk_label} (conf={confidence:.2f}, high-risk prob={high_risk_prob:.2f}). "
        f"Affected: {org_str}. Headline: {title}"
    )


def trace_article(article, components):
    title = article["title"]
    body = article["body"]
    text = f"{title} {body}"

    trace = {
        "article_id": article["article_id"],
        "title": title,
        "expected_alert": article.get("expected_alert"),
        "expected_orgs": article.get("expected_orgs", []),
        "sbert_loaded": components["sbert_filter"] is not None,
        "classifier_loaded": components["risk_classifier"] is not None,
        "crf_loaded": components["crf_extractor"] is not None and components["crf_extractor"].crf is not None,
        "llm_loaded": components["alert_generator"] is not None,
        "passed_sbert": None,
        "similarity_score": None,
        "risk_label": None,
        "risk_confidence": None,
        "high_risk_prob": None,
        "alert_triggered": False,
        "orgs": [],
        "alert_message": None,
        "root_error_stage": None,
        "error_effect": None,
    }

    sbert_filter = components["sbert_filter"]
    classifier = components["risk_classifier"]
    crf_extractor = components["crf_extractor"]
    alert_generator = components["alert_generator"]

    if sbert_filter is None:
        trace["root_error_stage"] = "environment"
        trace["error_effect"] = "SBERT filter could not be loaded."
        return trace

    passed, score = sbert_filter.check_similarity(title, body)
    trace["passed_sbert"] = passed
    trace["similarity_score"] = score

    if not passed:
        trace["alert_message"] = "Filtered: headline-body mismatch."
        if article.get("expected_alert") is True:
            trace["root_error_stage"] = "denoising"
            trace["error_effect"] = "False negative: valid article removed before classification."
        return trace

    if classifier is None:
        trace["root_error_stage"] = "environment"
        trace["error_effect"] = "Risk classifier could not be loaded after denoising."
        return trace

    risk_result = classifier.predict(text)
    trace["risk_label"] = risk_result["label"]
    trace["risk_confidence"] = risk_result["confidence"]
    trace["high_risk_prob"] = float(risk_result["probabilities"][0])

    trace["alert_triggered"] = (
        trace["risk_label"] == "High Risk" or trace["high_risk_prob"] >= HIGH_RISK_ALERT_THRESHOLD
    )

    if article.get("expected_alert") is True and not trace["alert_triggered"]:
        trace["root_error_stage"] = "classification"
        trace["error_effect"] = "False negative: risky article produced no alert."
    elif article.get("expected_alert") is False and trace["alert_triggered"]:
        trace["root_error_stage"] = "classification"
        trace["error_effect"] = "False positive: non-risk article triggered an alert."

    if trace["alert_triggered"] and crf_extractor is not None and crf_extractor.crf is not None:
        try:
            trace["orgs"] = crf_extractor.extract_orgs(text)
        except Exception as exc:
            trace["orgs"] = []
            if trace["root_error_stage"] is None:
                trace["root_error_stage"] = "ner"
                trace["error_effect"] = f"NER runtime failure prevented organization extraction: {exc}"

    if trace["alert_triggered"] and article.get("expected_orgs"):
        expected_orgs = set(article.get("expected_orgs", []))
        observed_orgs = set(trace["orgs"])
        if expected_orgs and not expected_orgs.intersection(observed_orgs) and trace["root_error_stage"] is None:
            trace["root_error_stage"] = "ner"
            trace["error_effect"] = "Alert fired, but NER missed the target organizations."

    if trace["alert_triggered"]:
        body_snippet = body[:200] + ("..." if len(body) > 200 else "")
        if alert_generator is not None:
            try:
                trace["alert_message"] = alert_generator.generate(
                    title=title,
                    risk_label=trace["risk_label"],
                    confidence=trace["risk_confidence"],
                    orgs=trace["orgs"],
                    body_snippet=body_snippet,
                )
            except Exception:
                trace["alert_message"] = fallback_alert_template(
                    title,
                    trace["risk_label"],
                    trace["risk_confidence"],
                    trace["orgs"],
                    trace["high_risk_prob"],
                )
        else:
            trace["alert_message"] = fallback_alert_template(
                title,
                trace["risk_label"],
                trace["risk_confidence"],
                trace["orgs"],
                trace["high_risk_prob"],
            )
    else:
        trace["alert_message"] = (
            f"No alert. Risk: {trace['risk_label']} (conf={trace['risk_confidence']:.3f})"
            if trace["risk_label"] is not None else "No alert."
        )

    return trace


## Representative Error-Analysis Set

These examples are intentionally chosen to stress different parts of the pipeline. They are adversarial, not random, so the goal is interpretability of failure modes rather than unbiased evaluation. They let us answer the questions that matter in an end-to-end system:

- Does a denoising error suppress a real alert before it can be classified?
- Does a classifier false positive generate a false alert?
- Does an NER miss produce an alert with missing affected organizations?
- When the LLM stage is unavailable, does the fallback template preserve the alert decision?

In [ ]:
articles = [
    {
        "article_id": "tp_high_risk",
        "title": "Apple faces regulatory pressure in China",
        "body": "Apple may face significant regulatory challenges in China next quarter as authorities tighten market rules and scrutiny over foreign technology firms rises.",
        "expected_alert": True,
        "expected_orgs": ["Apple"],
    },
    {
        "article_id": "tn_neutral",
        "title": "JPMorgan reports stronger quarterly profit on consumer banking growth",
        "body": "JPMorgan Chase reported quarterly earnings above analyst expectations, driven by consumer banking and stable credit performance. The company maintained full-year guidance.",
        "expected_alert": False,
        "expected_orgs": ["JPMorgan Chase"],
    },
    {
        "article_id": "fn_denoising",
        "title": "Markets panic after emergency liquidity warning",
        "body": "A regional bank disclosed acute liquidity pressure, but the article also contains several unrelated paragraphs and syndicated boilerplate that may weaken title-body similarity even though the core event is risky.",
        "expected_alert": True,
        "expected_orgs": [],
    },
    {
        "article_id": "fp_classifier",
        "title": "Tesla expands battery plant as executives warn of severe margin pressure and default risk across suppliers",
        "body": "Tesla said it will expand battery production after stronger deliveries and improved factory efficiency. Management also discussed severe margin pressure, refinancing risk, supplier distress, liquidity concerns, and possible defaults across weaker partners, but the overall article remains mainly positive corporate expansion news rather than a direct crisis at Tesla itself.",
        "expected_alert": False,
        "expected_orgs": ["Tesla"],
    },
    {
        "article_id": "ner_miss",
        "title": "Cyberattack disrupts settlement operations at major brokerage",
        "body": "The cyberattack disrupted settlement activity at Meridian Harbor Securities Clearing and Custody Holdings plc, interrupting client account access and increasing operational and reputational risk across the brokerage's post-trade infrastructure.",
        "expected_alert": True,
        "expected_orgs": ["Meridian Harbor Securities Clearing and Custody Holdings plc"],
    },
]

articles_df = pd.DataFrame(articles)
articles_df[["article_id", "title", "expected_alert", "expected_orgs"]]

In [ ]:
traces = [trace_article(article, components) for article in articles]
trace_df = pd.DataFrame(traces)

cols = [
    "article_id",
    "passed_sbert",
    "similarity_score",
    "risk_label",
    "risk_confidence",
    "high_risk_prob",
    "alert_triggered",
    "orgs",
    "root_error_stage",
    "error_effect",
]
trace_df[cols]

In [ ]:
def assign_propagation_label(row):
    if row["root_error_stage"] == "denoising":
        return "Missed alert caused upstream by denoising"
    if row["root_error_stage"] == "classification" and row["expected_alert"] is False and row["alert_triggered"] is True:
        return "False alert caused by classifier"
    if row["root_error_stage"] == "classification" and row["expected_alert"] is True and row["alert_triggered"] is False:
        return "Missed alert caused by classifier"
    if row["root_error_stage"] == "ner":
        return "Alert triggered, but target organization information degraded"
    if row["root_error_stage"] == "environment":
        return "Infrastructure prevented full pipeline execution"
    return "No clear propagated error in this example"


analysis_df = trace_df.copy()
analysis_df["propagation_summary"] = analysis_df.apply(assign_propagation_label, axis=1)

analysis_df[[
    "article_id",
    "expected_alert",
    "alert_triggered",
    "root_error_stage",
    "error_effect",
    "propagation_summary",
]]

## How to Read the Error Analysis

Interpret the trace table from left to right:

1. **Denoising failure**: if `passed_sbert=False` for an article that should alert, the article is dropped before any later stage can recover it. This is the strongest error-propagation case because the pipeline stops immediately.
2. **Classification failure**: if the article passes SBERT but the alert decision disagrees with `expected_alert`, then the classifier is the root cause. A false positive here becomes a direct false alert.
3. **NER failure**: if the alert triggers correctly but `orgs` misses expected organizations, the system still produces an alert, but the alert is incomplete or harder to action.
4. **Alert generation fallback**: if the LLM generator is unavailable, the final message becomes template-based. This changes fluency, but it should not change the alert decision itself.

In other words, the pipeline has a strict dependency chain:

`raw article -> denoising -> classification -> NER -> alert text`

Errors propagate forward, not backward. Earlier-stage mistakes are therefore usually more damaging than later-stage mistakes.

## Suggested Write-up Points for the Report

- The system architecture is deliberately conservative: denoising prevents noisy inputs from reaching later stages, and the alert threshold adds recall for borderline high-risk cases.
- The most costly failure mode is a denoising false negative, because the article never reaches classification or alerting.
- Classification errors directly control alert correctness. A classifier false positive produces a false alert; a classifier false negative suppresses a needed alert.
- NER errors are usually partial failures rather than total failures: the alert may still fire, but with missing or incorrect affected organizations.
- The final generation stage is relatively low risk because the implementation already falls back to a deterministic template when the LLM is unavailable.
- This notebook can be rerun on additional manually curated articles to expand the error taxonomy without retraining any model.